# DyNECT P2P — controlled reduction from the full IEEE33 game

**Goal:** Start with the original **48-hour, 30-minute** active-power-only `DynLQGameTV`. Inspect constraints, remove only constraints that are demonstrably redundant or local, solve the vGNE after **every** structural reduction, and **only then** aggregate to 2-hour intervals. At 2 hours, compare a **full-feeder** game with a **screened/localized-feeder** game and run the **original centralized solver** on the same aggregated data.

The exact reduction sequence is:

1. **Full 30-min model** with original active-only feeder → vGNE and saved-result check.
2. Clean exactly zero and identical local inequality rows → vGNE comparison.
3. **Certify redundant feeder limits** using conservative power-injection intervals → vGNE comparison.
4. Move feeder inequalities involving **only one prosumer** from shared to local constraints → vGNE comparison.
5. Optional **no-feeder diagnostic** (changes feasible set; never described as exact).
6. **Aggregate to 2-hour intervals**, rebuild **full feeder**, repeat exact reductions, and compare both 2-hour vGNE solutions.
7. Run **`algo_central.centralized` at 2 hours**, compare with the 2-hour full/screened DyNECT games.
8. Optional: test whether a **24-hour horizon** is adequate, and inspect GNE enumeration dimensions without launching it.

**Crucial boundaries:** no changes to your supervisor's DyNECT repository or your original P2P files; active-power-only is **already** an approximation of the original P/Q model; exact constraint reductions preserve that *active-only* game, not the original reactive model. Finite solver tolerances mean we check agreement rather than assert bitwise equality. A full GNE-set solver is not launched automatically.

## 0. Imports and one experiment configuration

In [1]:
import sys, json, copy, math, time
from pathlib import Path
from collections import Counter
from dataclasses import replace

import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'scenarios.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'scenarios.py').exists():
    raise FileNotFoundError('Open from the original P2P project root or a subfolder.')
sys.path.insert(0, str(ROOT))

# The separate DyNECT Git repository; use editable installation if available.
DYNECT_REPO = Path.home() / 'Documents/Github/dynect_py_p2p'
if DYNECT_REPO.exists():
    sys.path.insert(0, str(DYNECT_REPO))

import config, dynect
from dynect import DynLQGameTV
from scenarios import make_scenario_horizon
from p2p.setup import (build_game, load_ieee33, _feeder_maps,
                       unpack_controls, rollout_states, evaluated_costs,
                       feeder_results, to_static_gnep)
from p2p.vgne import solve_vgne

DATE = '2016-01-07'
SEED = 42
DT_FINE = 0.5
DT_COARSE = 2.0
LOOKAHEAD_H = 24    # First inspect full 48h. A 24h sensitivity test comes LAST.
TOL_COST = 1e-3     # EUR, comparative numerical tolerance
TOL_DISPATCH = 1e-3 # kW, comparative numerical tolerance

assert config.NETWORK_TOPOLOGY == 'ieee33', 'This feeder reducer supports IEEE33 only.'
assert config.PROSUMER_SCENARIO == 4, 'Begin with the original four-prosumer case.'
print('P2P root:', ROOT)
print('DyNECT:', dynect.__file__)
print('Experiment:', DATE, config.NETWORK_TOPOLOGY,
      config.PROSUMER_SCENARIO, config.TRADING_TOPOLOGY)

P2P root: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative
DyNECT: /Users/catharinahente/Documents/Github/dynect_py_p2p/dynect/__init__.py
Experiment: 2016-01-07 ieee33 4 cyclic


## 1. Load the full-resolution study inputs once; construct the FULL active-only game

This uses your original `make_scenario_horizon`, including the 24-hour lookahead. **Do not use 2-hour data yet.**

In [2]:
raw = make_scenario_horizon(DATE, seed=SEED, resolution_h=DT_FINE,
                            lookahead_h=LOOKAHEAD_H)
grid = load_ieee33()
full = build_game(raw, grid=grid, clearing_epsilon_kw=0.0)

print(f'Full: N={full.N}, T={full.T}, dt={raw["dt"]}h, '
      f'horizon={full.T*raw["dt"]:g}h, evaluation={raw["T_eval"]*raw["dt"]:g}h')
print('States:', full.game.nx, '| controls/agent/step:', full.game.nu)
print('Prosumer buses:', raw['agent_bus'], '| P2P edges:', raw['edges'])
print('Shared rows per step:', full.game.m_u,
      '| total:', full.T * full.game.m_u,
      '| beta dimension:', (full.N - 1) * full.T * full.game.m_u)

Building winter scenario (T=48, dt=0.5h, resolution=30min)...
  Prices 2016: loaded from cache
  P1: Residential ++ (bus 5)
    HP: Air_Alternative_1 (Air alternative, Hanover)
  P2: Office building (bus 11)
    ⚠ No EV activity detected — using fallback schedule
  P3: Residential HP only (bus 17)
    HP: Air_Semi-Parallel_1 (Air semi-parallel, Hanover)
  P4: Residential PV only (bus 32)
  Done. N=4, T=48, dt=0.5h, max_demand=16.3 kW
Building winter scenario (T=48, dt=0.5h, resolution=30min)...
  Prices 2016: loaded from cache
  P1: Residential ++ (bus 5)
    HP: Air_Alternative_1 (Air alternative, Hanover)
  P2: Office building (bus 11)
    ⚠ No EV activity detected — using fallback schedule
  P3: Residential HP only (bus 17)
    HP: Air_Semi-Parallel_1 (Air semi-parallel, Hanover)
  P4: Residential PV only (bus 32)
  Done. N=4, T=48, dt=0.5h, max_demand=16.8 kW
Full: N=4, T=96, dt=0.5h, horizon=48h, evaluation=24h
States: 7 | controls/agent/step: [11, 11, 11, 11]
Prosumer buses: [5, 

## 2. Inspect ALL constraint families before touching them

Rows are **shared** if they couple agents, **local** for individual inputs, and **state** for BESS/EV/HP. The 33-bus feeder is *already algebraically substituted* into inequalities in `p2p/setup.py`: there are no separate line-flow optimization variables to eliminate.

In [3]:
def size_report(case, name):
    g = case.game
    groups = Counter('trade reciprocity' if s.startswith('edge_') else
                     'feeder flow' if s.startswith('flow_') else
                     'feeder voltage' if s.startswith('voltage_') else 'other'
                     for s in case.shared_labels)
    out = {'Stage': name, 'steps': case.T,
           'decision variables': case.T * sum(g.nu),
           'shared rows': case.T * g.m_u,
           'beta dimension': (case.N-1) * case.T * g.m_u,
           'local rows': sum(g.C_loc[t][i].shape[0]
                             for t in range(case.T) for i in range(case.N)),
           'state rows': sum(g.C_x[t].shape[0] for t in range(case.T)),
           **{k+' rows': v*case.T for k,v in groups.items()}}
    return out

sizes = [size_report(full, 'Full 30 min')]
display(pd.DataFrame(sizes).fillna(0).set_index('Stage').T)

# Which participants can change each SHARED constraint? (structural, not from one solution)
rows = []
for k, label in enumerate(full.shared_labels):
    affected = [i+1 for i in range(full.N)
                if any(np.any(full.game.C_u[t][i][k] != 0)
                       for t in range(full.T))]
    rows.append({'Constraint': label, 'Agents affecting row': affected,
                 'Number of agents': len(affected)})
structural = pd.DataFrame(rows)
print('Shared-row ownership (per time-step label):')
display(structural.groupby('Number of agents').size().rename('Row labels').to_frame())
display(structural.head(12))

Stage,Full 30 min
steps,96
decision variables,4224
shared rows,13056
beta dimension,39168
local rows,12288
state rows,1350
trade reciprocity rows,768
feeder flow rows,6144
feeder voltage rows,6144


Shared-row ownership (per time-step label):


,Row labels
Number of agents,
0,14
1,28
2,20
4,74


,Constraint,Agents affecting row,Number of agents
0,edge_0_1_pos,"[1, 2]",2
1,edge_0_1_neg,"[1, 2]",2
2,edge_1_2_pos,"[2, 3]",2
3,edge_1_2_neg,"[2, 3]",2
4,edge_2_3_pos,"[3, 4]",2
5,edge_2_3_neg,"[3, 4]",2
6,edge_0_3_pos,"[1, 4]",2
7,edge_0_3_neg,"[1, 4]",2
8,"flow_(0, 1)_upper","[1, 2, 3, 4]",4
9,"flow_(0, 1)_lower","[1, 2, 3, 4]",4


## 3. Establish the FULL 30-minute vGNE benchmark

Solve once. We reuse this result for all subsequent comparisons. Saved centralized costs are independent, but the legacy solver includes reactive power; agreement is an empirical check, **not an identity guaranteed by construction**.

In [4]:
def solve_compact(case, verbose=False):
    """Solve the vGNE and release the huge condensed AVI/mpAVI immediately.

    Retain only dispatch, objective and diagnostics; otherwise repeated
    30-minute solves can exhaust notebook memory.
    """
    result = solve_vgne(case,verbose=verbose)
    result.pop('mp',None)
    result.pop('avi',None)
    return result

t0 = time.perf_counter()
v_full = solve_compact(full, verbose=False)
print('Gurobi status:', v_full['gurobi_status'],
      '| time [s]:', round(time.perf_counter()-t0,1),
      '| max AVI violation:', f'{v_full["max_violation"]:.2e}')
J_full = evaluated_costs(full, v_full['u'], first_day=True)
path = (ROOT/'structures'/f'{config.NETWORK_TOPOLOGY}_{full.N}_{config.TRADING_TOPOLOGY}'
        /'results'/DATE/'centralized.json')
comparison0 = pd.DataFrame({'Prosumer': [f'P{i+1}' for i in range(full.N)],
                            'vGNE first 24h [EUR]': J_full})
if path.is_file():
    saved_old = json.loads(path.read_text())
    J_old = np.asarray(saved_old['costs'],float)
    comparison0['Saved centralized first 24h [EUR]'] = J_old
    comparison0['Difference [EUR]'] = J_full - J_old
else:
    print('Saved centralized result not found:',path)
display(comparison0.round(5))

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2806500
Academic license 2806500 - for non-commercial use only - registered to ch___@ethz.ch
Gurobi status: 2 | time [s]: 9.3 | max AVI violation: 4.75e-14


,Prosumer,vGNE first 24h [EUR],Saved centralized first 24h [EUR],Difference [EUR]
0,P1,9.55805,9.55805,-0.0
1,P2,55.00790,55.00790,-0.0
2,P3,2.52852,2.52852,0.0
3,P4,10.88520,10.88520,-0.0


In [5]:
def shared_activity(case, u, threshold=1e-6):
    """Solution-specific slack diagnostics; NOT a redundancy certificate."""
    controls = unpack_controls(case,u)
    slack = np.vstack([
        case.game.b_u[t] - sum(
            case.game.C_u[t][i] @ np.array([
                controls[i][name][t] for name in case.controls[i]])
            for i in range(case.N))
        for t in range(case.T)])
    return pd.DataFrame({
        'Constraint':case.shared_labels,
        'Minimum slack':slack.min(axis=0),
        'Near-active time steps':(np.abs(slack) <= threshold).sum(axis=0)
    })

activity = shared_activity(full,v_full['u'])
print('Most nearly active shared constraint labels (native units, NOT proof of redundancy):')
display(activity.sort_values('Minimum slack').head(16).round(8))
print('Local input inequality rows per agent over the full horizon:')
display(pd.DataFrame({'Prosumer':[f'P{i+1}' for i in range(full.N)],
                      'Local rows':[sum(full.game.C_loc[t][i].shape[0]
                                        for t in range(full.T)) for i in range(full.N)]}))

Most nearly active shared constraint labels (native units, NOT proof of redundancy):


,Constraint,Minimum slack,Near-active time steps
0,edge_0_1_pos,-0.000000,96
1,edge_0_1_neg,-0.000000,96
2,edge_1_2_pos,-0.000000,96
3,edge_1_2_neg,-0.000000,96
4,edge_2_3_pos,-0.000000,96
5,edge_2_3_neg,-0.000000,96
6,edge_0_3_pos,-0.000000,96
7,edge_0_3_neg,-0.000000,96
105,voltage_17_lower,0.074288,0
103,voltage_16_lower,0.075114,0


Local input inequality rows per agent over the full horizon:


,Prosumer,Local rows
0,P1,3072
1,P2,3072
2,P3,3072
3,P4,3072


## 4. Helper functions: exact local cleanup, shared-row selection and audit

A zero row with a nonnegative RHS is tautological. Identical inequality normals can be kept once with the **tightest** RHS. Opposite rows encoding equality are **not** duplicates. We never delete an inequality merely because it is slack at one vGNE.

In [6]:
def clean_local(A, b):
    A = np.asarray(A, float)
    b = np.asarray(b, float)
    best = {}
    for row, rhs in zip(A, b):
        row = row.copy()
        row[row == 0.] = 0.  # normalize -0.0, no numerical rounding
        if not np.any(row):
            if rhs < 0:
                raise ValueError('Infeasible identically-zero local row')
            continue
        key = row.tobytes()   # only EXACT identical normals
        if key not in best or rhs < best[key][1]:
            best[key] = (row, rhs)
    if not best:
        return np.zeros((0, A.shape[1])), np.zeros(0)
    return np.vstack([v[0] for v in best.values()]), np.array([v[1] for v in best.values()])

GAME_FIELDS = ['A','B','Q','R','P','c','q','r','p',
               'C_x','b_x','C_loc','b_loc','C_u','b_u']

def rebuild_constraints(base, *, keep_labels=None, clean_locals=False,
                        localize_single_agent=False):
    """Create a new DynLQGameTV, preserving dynamics/costs and control ordering.

    Shared rows can be removed by their labels (after an independent proof of
    redundancy), or moved to local constraints when they involve one agent.
    """
    old = base.game
    kw = {k: getattr(old,k) for k in GAME_FIELDS}
    labels = list(base.shared_labels)
    if keep_labels is None:
        keep_labels = labels
    if len(set(keep_labels)) != len(keep_labels):
        raise ValueError('Shared label names must be unique')
    indices = [j for j,label in enumerate(labels) if label in set(keep_labels)]
    if len(indices) != len(keep_labels):
        raise ValueError('Unknown requested shared label')
    c_loc, b_loc, c_u, b_u = [], [], [], []
    kept_labels = [labels[j] for j in indices]
    moved = []
    ownership = {}
    if localize_single_agent:
        for j in indices:
            owners = [i for i in range(base.N) if any(
                np.any(old.C_u[t][i][j] != 0) for t in range(base.T))]
            ownership[j] = owners
            if len(owners) == 1:
                moved.append(j)
            elif len(owners) == 0:
                if any(old.b_u[t][j] < 0 for t in range(base.T)):
                    raise ValueError(f'Constant infeasible shared row: {labels[j]}')
                moved.append(j)
        indices = [j for j in indices if j not in moved]
        kept_labels = [labels[j] for j in indices]
    for t in range(base.T):
        local_a = [np.asarray(old.C_loc[t][i]).copy() for i in range(base.N)]
        local_b = [np.asarray(old.b_loc[t][i]).copy() for i in range(base.N)]
        if localize_single_agent:
            for j in moved:
                owners = ownership[j]
                if len(owners) == 1:
                    i = owners[0]
                    local_a[i] = np.vstack((local_a[i], old.C_u[t][i][j:j+1]))
                    local_b[i] = np.append(local_b[i], old.b_u[t][j])
        if clean_locals:
            cleaned = [clean_local(a,b) for a,b in zip(local_a,local_b)]
            local_a = [x[0] for x in cleaned]
            local_b = [x[1] for x in cleaned]
        c_loc.append(local_a); b_loc.append(local_b)
        c_u.append([old.C_u[t][i][indices,:].copy() for i in range(base.N)])
        b_u.append(old.b_u[t][indices].copy())
    kw.update(C_loc=c_loc, b_loc=b_loc, C_u=c_u, b_u=b_u)
    new = replace(base, game=DynLQGameTV(**kw), shared_labels=kept_labels)
    return new, [labels[j] for j in moved]

def model_max_violation(case, u):
    """Check a candidate against the given (possibly FULL) dynamic game's rows,
    without constructing an additional enormous condensed AVI.
    """
    controls = unpack_controls(case, u)
    states = rollout_states(case, u)
    worst = -np.inf
    for t in range(case.T):
        shared = np.zeros(len(case.game.b_u[t]))
        for i in range(case.N):
            ui = np.array([controls[i][name][t] for name in case.controls[i]])
            shared += case.game.C_u[t][i] @ ui
            A = case.game.C_loc[t][i]
            if A.shape[0]:
                worst = max(worst,float(np.max(A @ ui - case.game.b_loc[t][i])))
        if shared.size:
            worst = max(worst,float(np.max(shared-case.game.b_u[t])))
        if case.game.C_x[t].shape[0]:
            worst = max(worst,float(np.max(case.game.C_x[t]@states[t+1]-case.game.b_x[t])))
    return max(0.,float(worst))

history = []
def check_solution(name, case, sol, ref_case=None, ref_sol=None):
    J = evaluated_costs(case, sol['u'], first_day=True)
    row = {'Stage':name,'Total day-1 cost [EUR]':float(J.sum()),
           'Max own violation':model_max_violation(case,sol['u']),
           'Shared rows':case.T*case.game.m_u,
           'Beta dimension':(case.N-1)*case.T*case.game.m_u}
    if ref_sol is not None:
        if case.T != ref_case.T or case.game.nu != ref_case.game.nu:
            raise ValueError('Direct decision comparisons require the same time grid and ordering')
        J_ref = evaluated_costs(ref_case,ref_sol['u'],first_day=True)
        row['Max cost Δ [EUR]'] = float(np.max(np.abs(J-J_ref)))
        row['Control RMS Δ [kW]'] = float(np.sqrt(np.mean((sol['u']-ref_sol['u'])**2)))
        row['Max FULL-model violation'] = model_max_violation(ref_case,sol['u'])
        # The cost quadratic is unchanged by constraint-only reductions.
        row['Full potential objective Δ [EUR]'] = float(sol['objective']-ref_sol['objective'])
        row['Coincident costs?'] = row['Max cost Δ [EUR]'] <= TOL_COST
        row['Coincident dispatch?'] = row['Control RMS Δ [kW]'] <= TOL_DISPATCH
    history.append(row)
    display(pd.DataFrame([row]).set_index('Stage').round(7))
    return J

check_solution('Full 30 min',full,v_full)

,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension
Stage,,,,
Full 30 min,77.97967,0.0,13056,39168


array([ 9.55805355, 55.00789929,  2.52851586, 10.88520121])

## 5. EXACT reduction A: clean local input inequalities, then re-solve

Identical normals, including any repeated bounds, are merged. State constraints are left untouched. This should preserve all feasible decisions exactly.

In [7]:
local_clean, _ = rebuild_constraints(full, clean_locals=True)
sizes.append(size_report(local_clean,'Clean local 30 min'))
print('Local rows:', sizes[0]['local rows'],'→',sizes[-1]['local rows'])
v_local = solve_compact(local_clean,verbose=False)
check_solution('Clean local 30 min',local_clean,v_local,full,v_full)

Local rows: 12288 → 12288


,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension,Max cost Δ [EUR],Control RMS Δ [kW],Max FULL-model violation,Full potential objective Δ [EUR],Coincident costs?,Coincident dispatch?
Stage,,,,,,,,,,
Clean local 30 min,77.97967,0.0,13056,39168,0.0,0.0,0.0,0.0,True,True


array([ 9.55805355, 55.00789929,  2.52851586, 10.88520121])

## 6. Identify feeder limits that are **provably redundant**

We compute a conservative **independent injection interval** for every prosumer and time step using demand, PV, BESS/EV converter limits, EV availability and HP bounds. This envelope is intentionally broader than the true feasible set, since SOC and local energy balance are omitted. If a feeder inequality is satisfied throughout the broader envelope, removing it is safe. A constraint being slack at the vGNE **alone is not sufficient**.

In [8]:
def injection_intervals(data):
    N, T = len(data['demand']), len(data['buy_price'])
    lo, hi = np.zeros((N,T)), np.zeros((N,T))
    base = np.asarray(data['generation'],float) - np.asarray(data['demand'],float)
    available = np.asarray(data['ev_availability'],float)
    for i in range(N):
        batt_limit = min(float(data['battery_power'][i]),
                         float(data['battery_smax'][i]) *
                         math.cos(math.pi/float(data.get('battery_polygon_sides',8))))
        ev_limit = min(float(data['ev_power'][i]),
                       float(data['ev_smax'][i]) *
                       math.cos(math.pi/float(data.get('ev_polygon_sides',8))))
        ev_ch = ev_limit*available[i]
        ev_dis = ev_ch * float(bool(data.get('ev_v2g',True)))
        lo[i] = base[i] - batt_limit - ev_ch - float(data['hp_max_power'][i])
        hi[i] = base[i] + batt_limit + ev_dis
    return lo, hi

def certify_feeder_constraints(case, margin=1e-8):
    """Safe global screening: returns per-shared-label interval upper bound on
    maximum constraint violation. Only negative *certified* bounds are removed.
    MW and squared-pu rows are compared in their native units.
    """
    if case.grid is None:
        raise ValueError('Need original feeder to certify feasibility')
    g, d = case.grid, case.data
    lo,hi = injection_intervals(d)
    f0,L,drop = _feeder_maps(g,list(map(int,d['agent_bus'])))
    a = L[:,:,None] * lo[None,:,:]
    b = L[:,:,None] * hi[None,:,:]
    flow_min = f0[:,None] + np.minimum(a,b).sum(axis=1)
    flow_max = f0[:,None] + np.maximum(a,b).sum(axis=1)
    vm = -drop @ L
    v0 = float(g['root_voltage'])**2 - drop @ f0
    va = vm[:,:,None]*lo[None,:,:]
    vb = vm[:,:,None]*hi[None,:,:]
    v_min = v0[:,None] + np.minimum(va,vb).sum(axis=1)
    v_max = v0[:,None] + np.maximum(va,vb).sum(axis=1)
    lim = float(d['line_limit_mva'])*math.cos(math.pi/8)
    bound = {}
    for k,e in enumerate(g['branches']):
        bound[f'flow_{e}_upper'] = float(np.max(flow_max[k] - lim))
        bound[f'flow_{e}_lower'] = float(np.max(-lim-flow_min[k]))
    for bus in range(g['bus_count']):
        if bus == g['root']:
            continue
        bound[f'voltage_{bus}_upper'] = float(np.max(v_max[bus]-g['voltage_max'][bus]**2))
        bound[f'voltage_{bus}_lower'] = float(np.max(g['voltage_min'][bus]**2-v_min[bus]))
    records = []
    for label in case.shared_labels:
        if label.startswith('edge_'):
            records.append({'Label':label,'Type':'P2P',
                            'Worst possible violation (bound)':np.nan,
                            'Certified redundant':False})
        else:
            if label not in bound:
                raise KeyError(f'Unknown feeder label: {label}')
            records.append({'Label':label,'Type':'flow' if label.startswith('flow_') else 'voltage',
                            'Worst possible violation (bound)':bound[label],
                            'Certified redundant':bound[label] < -margin})
    return pd.DataFrame(records)

screen_report = certify_feeder_constraints(local_clean)
print('Certified redundant labels over ALL 96 steps:')
display(screen_report.groupby(['Type','Certified redundant']).size().rename('Labels').to_frame())
print('Most nearly binding or uncertified feeder labels (positive = may bind):')
display(screen_report[screen_report.Type!='P2P']
        .sort_values('Worst possible violation (bound)',ascending=False).head(14).round(8))
keep = screen_report.loc[~screen_report['Certified redundant'],'Label'].tolist()
print('Shared rows per step:',len(local_clean.shared_labels),'→',len(keep))

Certified redundant labels over ALL 96 steps:


,,Labels
Type,Certified redundant,
P2P,False,8
flow,True,64
voltage,True,64


Most nearly binding or uncertified feeder labels (positive = may bind):


,Label,Type,Worst possible violation (bound),Certified redundant
105,voltage_17_lower,voltage,-0.071238,True
103,voltage_16_lower,voltage,-0.072138,True
101,voltage_15_lower,voltage,-0.074688,True
99,voltage_14_lower,voltage,-0.076723,True
97,voltage_13_lower,voltage,-0.078777,True
95,voltage_12_lower,voltage,-0.081470,True
135,voltage_32_lower,voltage,-0.088273,True
133,voltage_31_lower,voltage,-0.088570,True
131,voltage_30_lower,voltage,-0.089654,True
93,voltage_11_lower,voltage,-0.089869,True


Shared rows per step: 136 → 8


## 7. EXACT reduction B: remove certified redundant feeder inequalities

Every removed network inequality has a conservative envelope strictly within its bound for the **entire 48-hour window**. The original full model remains available for cross-checking.

In [9]:
screened, _ = rebuild_constraints(local_clean,keep_labels=keep,clean_locals=True)
sizes.append(size_report(screened,'Feeder screened 30 min'))
v_screened = solve_compact(screened,verbose=False)
check_solution('Feeder screened 30 min',screened,v_screened,full,v_full)

,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension,Max cost Δ [EUR],Control RMS Δ [kW],Max FULL-model violation,Full potential objective Δ [EUR],Coincident costs?,Coincident dispatch?
Stage,,,,,,,,,,
Feeder screened 30 min,77.97967,0.0,768,2304,0.0,0.0,0.0,0.0,True,True


array([ 9.55805355, 55.00789929,  2.52851586, 10.88520121])

## 8. EXACT reduction C: move single-agent feeder inequalities into LOCAL rows

If a shared inequality has nonzero coefficients in **only one player’s control block**, it is really an individual restriction; moving it to that player’s local constraints preserves everyone’s unilateral feasible set and hence the GNE set. Constant feasible shared rows are removed. The P2P clearing inequalities still couple both agents on each edge and must remain shared.

In [10]:
localized, moved = rebuild_constraints(screened,clean_locals=True,
                                       localize_single_agent=True)
sizes.append(size_report(localized,'Localized 30 min'))
print('Shared labels moved/removed:',len(moved))
print('First moved labels:',moved[:15])
print('Shared rows per step:',screened.game.m_u,'→',localized.game.m_u)
print('Beta dimension:',(localized.N-1)*localized.T*localized.game.m_u)
v_localized = solve_compact(localized,verbose=False)
check_solution('Localized 30 min',localized,v_localized,full,v_full)

Shared labels moved/removed: 0
First moved labels: []
Shared rows per step: 8 → 8
Beta dimension: 2304


,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension,Max cost Δ [EUR],Control RMS Δ [kW],Max FULL-model violation,Full potential objective Δ [EUR],Coincident costs?,Coincident dispatch?
Stage,,,,,,,,,,
Localized 30 min,77.97967,0.0,768,2304,0.0,0.0,0.0,0.0,True,True


array([ 9.55805355, 55.00789929,  2.52851586, 10.88520121])

## 9. Optional diagnostic — remove **all** feeder inequalities

This is NOT an exact reduction unless the no-feeder optimum also satisfies the full constraints and produces the same optimum. It helps isolate the actual influence of IEEE33; this is a **separate, relaxed game**. The original voltage and flow checks still run against the full feeder.

In [11]:
RUN_NO_FEEDER_DIAGNOSTIC = False
if RUN_NO_FEEDER_DIAGNOSTIC:
    trade_only = [s for s in localized.shared_labels if s.startswith('edge_')]
    # Start BEFORE feeder rows were moved into local constraints.
    # Otherwise these localized network constraints would remain.
    no_feeder, _ = rebuild_constraints(local_clean,keep_labels=trade_only,clean_locals=True)
    v_no_feeder = solve_compact(no_feeder,verbose=False)
    check_solution('No feeder 30 min (approx.)',no_feeder,v_no_feeder,full,v_full)
    display(pd.DataFrame([feeder_results(full,v_no_feeder['u'])])[
        ['max_abs_line_mw','min_voltage_pu','max_voltage_pu',
         'max_congestion_mw','max_voltage_violation_pu']])
else:
    print('Skipped. Set RUN_NO_FEEDER_DIAGNOSTIC=True to assess sensitivity.')

Skipped. Set RUN_NO_FEEDER_DIAGNOSTIC=True to assess sensitivity.


## 10. Size and sparsity audit after structural reductions

`scipy.sparse` gives a useful representation of individual constraint matrices, but **the supplied DyNECT conversion still constructs dense arrays**. These numbers describe *potential* storage savings; wrapping in CSR alone does **not** magically make `MpAVI.from_game` sparse.

In [12]:
display(pd.DataFrame(sizes).fillna(0).set_index('Stage').T)

def stage_sparsity(case):
    mats = [np.hstack([case.game.C_u[t][i] for i in range(case.N)])
            for t in range(case.T)]
    a = sparse.block_diag(mats,format='csr')
    dense_bytes = sum(m.nbytes for m in mats)
    csr_bytes = a.data.nbytes + a.indices.nbytes + a.indptr.nbytes
    return {'Stage': f'{case.T} steps; {case.game.m_u} shared rows/step',
            'Shape':str(a.shape),'Nonzeros':a.nnz,'Density %':100*a.nnz/(a.shape[0]*a.shape[1])
            if a.shape[0]*a.shape[1] else 0,
            'Dense stage blocks [MB]':dense_bytes/1e6,
            'CSR block diagonal [MB]':csr_bytes/1e6}

display(pd.DataFrame([stage_sparsity(full),stage_sparsity(localized)]).round(3))
print('Reactive-power control names:',[name for ctl in localized.controls
                                       for name in ctl if name.startswith('q_')])
print('No Q/kVAR flow variables exist in the dynamic control vector or feeder constraints.')
print('Note: game.Q is the STATE-COST weighting matrix, not reactive power.')

Stage,Full 30 min,Clean local 30 min,Feeder screened 30 min,Localized 30 min
steps,96.0,96.0,96.0,96.0
decision variables,4224.0,4224.0,4224.0,4224.0
shared rows,13056.0,13056.0,768.0,768.0
beta dimension,39168.0,39168.0,2304.0,2304.0
local rows,12288.0,12288.0,12288.0,12288.0
state rows,1350.0,1350.0,1350.0,1350.0
trade reciprocity rows,768.0,768.0,768.0,768.0
feeder flow rows,6144.0,6144.0,0.0,0.0
feeder voltage rows,6144.0,6144.0,0.0,0.0


,Stage,Shape,Nonzeros,Density %,Dense stage blocks [MB],CSR block diagonal [MB]
0,96 steps; 136 shared rows/step,"(13056, 4224)",574464,1.042,4.596,6.946
1,96 steps; 8 shared rows/step,"(768, 4224)",33792,1.042,0.270,0.409


Reactive-power control names: []
No Q/kVAR flow variables exist in the dynamic control vector or feeder constraints.
Note: game.Q is the STATE-COST weighting matrix, not reactive power.


## 11. Only now: aggregate 30-minute inputs into **2-hour** intervals over the SAME 48 hours

Power (demand/PV) and prices are averaged; EV driving **energy** is summed; BESS/EV/HP coefficients are rebuilt with `dt=2`. A coarse interval is marked EV-available only if the EV is present for its **entire** duration. Mixed bins are counted; the coarse model cannot preserve sub-hourly EV departure timing exactly, so this is an approximation. **Do not compare 2-hour trajectories component by component against 30-minute trajectories.**

In [13]:
def aggregate_time_grid(data, target_dt=2.0):
    d = copy.deepcopy(data)
    old_dt = float(d['dt'])
    factor = int(round(target_dt/old_dt))
    if factor < 1 or not np.isclose(factor*old_dt,target_dt):
        raise ValueError('New dt must be an integer multiple of original dt')
    oldT = len(d['buy_price'])
    if oldT % factor or int(d['T_eval']) % factor:
        raise ValueError('Horizon and evaluation length must be divisible by aggregation')
    N = len(d['demand']); T=oldT//factor
    def agent_agg(key,op):
        a = np.asarray(d[key],float).reshape(N,T,factor)
        d[key]=a.mean(axis=2) if op=='mean' else a.sum(axis=2)
    for k in ('demand','generation','hp_base_power','hp_q_heat'):
        agent_agg(k,'mean')
    agent_agg('ev_driving_load','sum')   # EV trip consumption is ENERGY per bin
    availability=np.asarray(d['ev_availability'],int).reshape(N,T,factor)
    mixed=((availability.min(axis=2) != availability.max(axis=2)))
    d['ev_availability']=availability.min(axis=2).astype(int)
    for k in ('buy_price','sell_price','p2p_price'):
        d[k]=np.asarray(d[k],float).reshape(T,factor).mean(axis=1)
    d['dt']=float(target_dt);d['resolution_h']=float(target_dt)
    d['T']=T;d['T_eval']=int(data['T_eval'])//factor
    return d, mixed

coarse, mixed_bins = aggregate_time_grid(raw,DT_COARSE)
assert np.allclose(np.sum(raw['ev_driving_load'],axis=1),
                   np.sum(coarse['ev_driving_load'],axis=1))
print('48h time steps:',len(raw['buy_price']),'→',len(coarse['buy_price']))
print('24h evaluation steps:',raw['T_eval'],'→',coarse['T_eval'])
print('Mixed-availability EV bins (conservative no-charge):',int(mixed_bins.sum()))
if np.any(mixed_bins):
    print('WARNING: EV scheduling and departure timing changed; treat 2h results as an approximation.')

48h time steps: 96 → 24
24h evaluation steps: 48 → 12
Mixed-availability EV bins (conservative no-charge): 3


## 12. At 2 hours: FULL feeder vGNE (new reference)

This is **not** a comparison to the original saved 30-minute result. We first establish a new full-constraint benchmark at exactly the same 2-hour resolution, before reducing anything.

In [14]:
coarse_full = build_game(coarse,grid=grid,clearing_epsilon_kw=0.0)
print(size_report(coarse_full,'Full feeder 2h'))
v_coarse_full = solve_compact(coarse_full,verbose=False)
check_solution('Full feeder 2h',coarse_full,v_coarse_full)

{'Stage': 'Full feeder 2h', 'steps': 24, 'decision variables': 1056, 'shared rows': 3264, 'beta dimension': 9792, 'local rows': 3072, 'state rows': 342, 'trade reciprocity rows': 192, 'feeder flow rows': 1536, 'feeder voltage rows': 1536}


,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension
Stage,,,,
Full feeder 2h,78.377344,0.0,3264,9792


array([ 9.83541874, 54.98217065,  2.67677285, 10.88298185])

## 13. At 2 hours: repeat exact cleanup, interval screening and shared-to-local relocation; compare EACH vGNE

In [15]:
coarse_local,_ = rebuild_constraints(coarse_full,clean_locals=True)
v_coarse_local = solve_compact(coarse_local,verbose=False)
check_solution('Clean local 2h',coarse_local,v_coarse_local,coarse_full,v_coarse_full)

coarse_report = certify_feeder_constraints(coarse_local)
print('2h feeder screening:')
display(coarse_report.groupby(['Type','Certified redundant']).size().rename('Labels').to_frame())
coarse_keep = coarse_report.loc[~coarse_report['Certified redundant'],'Label'].tolist()
coarse_screened,_ = rebuild_constraints(coarse_local,keep_labels=coarse_keep,
                                         clean_locals=True)
v_coarse_screened = solve_compact(coarse_screened,verbose=False)
check_solution('Feeder screened 2h',coarse_screened,v_coarse_screened,
               coarse_full,v_coarse_full)

coarse_reduced,coarse_moved = rebuild_constraints(coarse_screened,clean_locals=True,
                                                   localize_single_agent=True)
v_coarse_reduced = solve_compact(coarse_reduced,verbose=False)
check_solution('Reduced feeder 2h',coarse_reduced,v_coarse_reduced,
               coarse_full,v_coarse_full)
print('Shared rows at 2h:',coarse_full.game.m_u,'→',coarse_reduced.game.m_u,
      '| moved single-agent labels:',len(coarse_moved))
display(pd.DataFrame([size_report(coarse_full,'Full feeder 2h'),
                      size_report(coarse_reduced,'Reduced feeder 2h')]).set_index('Stage').T)

,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension,Max cost Δ [EUR],Control RMS Δ [kW],Max FULL-model violation,Full potential objective Δ [EUR],Coincident costs?,Coincident dispatch?
Stage,,,,,,,,,,
Clean local 2h,78.377344,0.0,3264,9792,0.0,0.0,0.0,0.0,True,True


2h feeder screening:


,,Labels
Type,Certified redundant,
P2P,False,8
flow,True,64
voltage,True,64


,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension,Max cost Δ [EUR],Control RMS Δ [kW],Max FULL-model violation,Full potential objective Δ [EUR],Coincident costs?,Coincident dispatch?
Stage,,,,,,,,,,
Feeder screened 2h,78.377344,0.0,192,576,0.0,0.0,0.0,0.0,True,True


,Total day-1 cost [EUR],Max own violation,Shared rows,Beta dimension,Max cost Δ [EUR],Control RMS Δ [kW],Max FULL-model violation,Full potential objective Δ [EUR],Coincident costs?,Coincident dispatch?
Stage,,,,,,,,,,
Reduced feeder 2h,78.377344,0.0,192,576,0.0,0.0,0.0,0.0,True,True


Shared rows at 2h: 136 → 8 | moved single-agent labels: 0


Stage,Full feeder 2h,Reduced feeder 2h
steps,24.0,24.0
decision variables,1056.0,1056.0
shared rows,3264.0,192.0
beta dimension,9792.0,576.0
local rows,3072.0,3072.0
state rows,342.0,342.0
trade reciprocity rows,192.0,192.0
feeder flow rows,1536.0,NaN
feeder voltage rows,1536.0,NaN


## 14. Independent lower-resolution centralized comparison (ORIGINAL Gurobi solver)

Call `algo_central.centralized(coarse)` using exactly the same **2-hour** day, profiles, EV aggregation, 48h horizon and `T_eval=12`. It retains the **original reactive-power model**, whereas DyNECT uses active-only LinDistFlow, so agreement is a measured result—not mathematically guaranteed. The comparison between **full-feeder and reduced-feeder DyNECT** is the rigorous internal reduction check.

In [16]:
from algo_central import centralized

RUN_CENTRAL_2H = True  # Set False only to inspect model sizes without solving
if RUN_CENTRAL_2H:
    t0 = time.perf_counter()
    centralized_2h = centralized(copy.deepcopy(coarse))
    print('Original centralized 2h solve [s]:',round(time.perf_counter()-t0,1))
    J_cent2 = np.asarray(centralized_2h['costs'],float)
    J_cfull = evaluated_costs(coarse_full,v_coarse_full['u'],first_day=True)
    J_cred = evaluated_costs(coarse_reduced,v_coarse_reduced['u'],first_day=True)
    cdf = pd.DataFrame({
        'Prosumer':[f'P{i+1}' for i in range(coarse_full.N)],
        'Centralized original 2h [EUR]':J_cent2,
        'DyNECT full feeder 2h [EUR]':J_cfull,
        'DyNECT reduced feeder 2h [EUR]':J_cred,
        'Reduced minus full [EUR]':J_cred-J_cfull,
        'Full minus original [EUR]':J_cfull-J_cent2})
    display(cdf.round(5))
    print('Full vs reduced vGNE cost max delta [EUR]:',np.max(np.abs(J_cred-J_cfull)))
    print('Original centralized vs DyNECT full cost max delta [EUR]:',np.max(np.abs(J_cent2-J_cfull)))
    # Compare physical dispatch with the legacy centralized result where present.
    disp = unpack_controls(coarse_full,v_coarse_full['u'])
    trajectory_rows=[]
    for key in ('buy','sell','charge','discharge','ev_charge','ev_discharge','hp_power'):
        if key in centralized_2h:
            legacy = np.asarray(centralized_2h[key],float)
            current = np.vstack([d[key] for d in disp])
            if legacy.shape == current.shape:
                trajectory_rows.append({'Variable':key,
                    'RMS difference [kW]':float(np.sqrt(np.mean((legacy-current)**2))),
                    'Max abs difference [kW]':float(np.max(abs(legacy-current)))})
    if trajectory_rows:
        print('Original centralized vs DyNECT full physical trajectories:')
        display(pd.DataFrame(trajectory_rows).round(5))
else:
    print('Centralized 2h solve skipped.')

Original centralized 2h solve [s]: 0.1


,Prosumer,Centralized original 2h [EUR],DyNECT full feeder 2h [EUR],DyNECT reduced feeder 2h [EUR],Reduced minus full [EUR],Full minus original [EUR]
0,P1,9.83542,9.83542,9.83542,0.0,-0.0
1,P2,54.98217,54.98217,54.98217,0.0,-0.0
2,P3,2.67677,2.67677,2.67677,0.0,0.0
3,P4,10.88298,10.88298,10.88298,0.0,-0.0


Full vs reduced vGNE cost max delta [EUR]: 0.0
Original centralized vs DyNECT full cost max delta [EUR]: 8.801265494184918e-09
Original centralized vs DyNECT full physical trajectories:


,Variable,RMS difference [kW],Max abs difference [kW]
0,buy,0.0,0.0
1,sell,0.0,0.0
2,charge,0.0,0.0
3,discharge,0.0,0.0
4,ev_charge,0.0,0.0
5,ev_discharge,0.0,0.0
6,hp_power,0.0,0.0


## 15. Optional sensitivity: does the extra 24-hour lookahead actually matter?

Here we keep the **2-hour resolution** and truncate the already aggregated data from 48h to **24h**. SOC/terminal conditions now apply at a different endpoint; this changes the optimization game. Compare the **first 24h evaluated costs**, not the optimization objectives or dimensions. EV trip/departure constraints near midnight deserve particular attention.

In [17]:
def first_day_only(data):
    d=copy.deepcopy(data)
    T1=int(d['T_eval'])
    for key in ['demand','generation','ev_availability','ev_driving_load',
                'hp_base_power','hp_q_heat']:
        d[key]=np.asarray(d[key])[:,:T1].copy()
    for key in ['buy_price','sell_price','p2p_price']:
        d[key]=np.asarray(d[key])[:T1].copy()
    d['T']=T1
    d['T_eval']=T1
    d['lookahead_h']=0
    d['evaluation_h']=24
    return d

RUN_24H_SENSITIVITY = True
if RUN_24H_SENSITIVITY:
    short_data = first_day_only(coarse)
    short_full = build_game(short_data,grid=grid,clearing_epsilon_kw=0.0)
    v_short_full = solve_compact(short_full,verbose=False)
    short_report = certify_feeder_constraints(short_full)
    short_keep = short_report.loc[~short_report['Certified redundant'],'Label'].tolist()
    short_screened,_ = rebuild_constraints(short_full,keep_labels=short_keep,clean_locals=True)
    short_reduced,_ = rebuild_constraints(short_screened,clean_locals=True,
                                           localize_single_agent=True)
    v_short_reduced = solve_compact(short_reduced,verbose=False)
    J_short_full = evaluated_costs(short_full,v_short_full['u'],first_day=True)
    J_short_red = evaluated_costs(short_reduced,v_short_reduced['u'],first_day=True)
    J_48hour = evaluated_costs(coarse_full,v_coarse_full['u'],first_day=True)
    display(pd.DataFrame({'Prosumer':[f'P{i+1}' for i in range(short_full.N)],
                          '48h lookahead, day-1 cost':J_48hour,
                          '24h horizon, day-1 cost':J_short_full,
                          '24h reduced, day-1 cost':J_short_red}).round(4))
    print('24h reduced/full worst cost difference:',np.max(abs(J_short_red-J_short_full)))
    print('24h vs 48h first-day cost difference:',np.max(abs(J_short_full-J_48hour)))
else:
    print('24h horizon sensitivity skipped; set RUN_24H_SENSITIVITY=True.')

,Prosumer,"48h lookahead, day-1 cost","24h horizon, day-1 cost","24h reduced, day-1 cost"
0,P1,9.8354,9.8354,9.8354
1,P2,54.9822,54.9822,54.9822
2,P3,2.6768,2.6768,2.6768
3,P4,10.8830,10.8830,10.8830


24h reduced/full worst cost difference: 0.0
24h vs 48h first-day cost difference: 2.3612445332332754e-10


In [18]:
from p2p.setup import unpack_controls, evaluated_costs
import numpy as np
import pandas as pd

comparisons = [
    ("48h vs 24h — full feeder",
     coarse_full, v_coarse_full, short_full, v_short_full),
    ("48h vs 24h — reduced",
     coarse_reduced, v_coarse_reduced, short_reduced, v_short_reduced),
    ("24h full vs reduced",
     short_full, v_short_full, short_reduced, v_short_reduced),
]

rows = []

for name, case_a, sol_a, case_b, sol_b in comparisons:
    a = unpack_controls(case_a, sol_a["u"])
    b = unpack_controls(case_b, sol_b["u"])

    for i in range(case_a.N):
        for variable in a[i]:
            difference = (
                np.asarray(a[i][variable][:case_b.T])
                - np.asarray(b[i][variable])
            )

            rows.append({
                "Comparison": name,
                "Prosumer": f"P{i+1}",
                "Variable": variable,
                "RMS Δ": np.sqrt(np.mean(difference**2)),
                "Max |Δ|": np.max(np.abs(difference)),
            })

dispatch_check = pd.DataFrame(rows)

display(
    dispatch_check.groupby("Comparison")
    .agg(
        Max_absolute_difference=("Max |Δ|", "max"),
        Worst_variable_RMS=("RMS Δ", "max"),
    ).round(6)
)

print("Largest individual discrepancies:")
display(
    dispatch_check.nlargest(12, "Max |Δ|").round(6)
)

print("\nMaximum individual cost differences [EUR]:")
for name, case_a, sol_a, case_b, sol_b in comparisons:
    Ja = evaluated_costs(case_a, sol_a["u"])
    Jb = evaluated_costs(case_b, sol_b["u"])
    print(f"{name}: {np.max(np.abs(Ja - Jb)):.8f}")

,Max_absolute_difference,Worst_variable_RMS
Comparison,,
24h full vs reduced,0.0,0.0
48h vs 24h — full feeder,0.0,0.0
48h vs 24h — reduced,0.0,0.0


Largest individual discrepancies:


,Comparison,Prosumer,Variable,RMS Δ,Max |Δ|
22,48h vs 24h — full feeder,P3,buy,0.0,0.0
66,48h vs 24h — reduced,P3,buy,0.0,0.0
41,48h vs 24h — full feeder,P4,trade_abs_0,0.0,0.0
85,48h vs 24h — reduced,P4,trade_abs_0,0.0,0.0
10,48h vs 24h — full feeder,P1,trade_abs_3,0.0,0.0
54,48h vs 24h — reduced,P1,trade_abs_3,0.0,0.0
30,48h vs 24h — full feeder,P3,trade_abs_1,0.0,0.0
74,48h vs 24h — reduced,P3,trade_abs_1,0.0,0.0
21,48h vs 24h — full feeder,P2,trade_abs_2,0.0,0.0
65,48h vs 24h — reduced,P2,trade_abs_2,0.0,0.0



Maximum individual cost differences [EUR]:
48h vs 24h — full feeder: 0.00000000
48h vs 24h — reduced: 0.00000000
24h full vs reduced: 0.00000000


In [19]:
from p2p.setup import build_game
from collections import Counter

EPS = 0.01  # kW: approximate trade reciprocity

# Clip the existing 2h data to the first 24 hours
short_data = first_day_only(coarse)

# Rebuild with relaxed P2P clearing
eps_full = build_game(
    short_data,
    grid=grid,
    clearing_epsilon_kw=EPS,
)

# Apply the same certified feeder reduction
eps_report = certify_feeder_constraints(eps_full)
eps_keep = eps_report.loc[
    ~eps_report["Certified redundant"], "Label"
].tolist()

eps_screened, _ = rebuild_constraints(
    eps_full,
    keep_labels=eps_keep,
    clean_locals=True,
)

gne_case, moved = rebuild_constraints(
    eps_screened,
    clean_locals=True,
    localize_single_agent=True,
)

g = gne_case.game
N, T = gne_case.N, gne_case.T

m_sh = T * g.m_u
n_beta = (N - 1) * m_sh

print("=== GNE PROBLEM DIMENSIONS ===")
print(f"Prosumers:               {N}")
print(f"Horizon:                 {T * short_data['dt']:.0f} hours")
print(f"Time resolution:         {short_data['dt']} hours")
print(f"Time steps:              {T}")
print(f"Decision variables:      {T * sum(g.nu)}")
print(f"State variables:         {g.nx}")
print(f"Shared constraints:      {m_sh}")
print(f"Local constraints:       {sum(g.C_loc[t][i].shape[0] for t in range(T) for i in range(N))}")
print(f"State constraints:       {sum(g.C_x[t].shape[0] for t in range(T))}")
print(f"β dimension:             {n_beta}")
print(f"Trade tolerance:         ±{EPS} kW")

print("\nRemaining shared constraint types:")
display(pd.Series(
    dict(Counter(
        "Trade reciprocity" if s.startswith("edge_") else "Other"
        for s in gne_case.shared_labels
    ))
).mul(T).rename("Rows").to_frame())

assert n_beta <= 300, "β dimension exceeds exploratory limit"

=== GNE PROBLEM DIMENSIONS ===
Prosumers:               4
Horizon:                 24 hours
Time resolution:         2.0 hours
Time steps:              12
Decision variables:      528
State variables:         7
Shared constraints:      96
Local constraints:       1536
State constraints:       174
β dimension:             288
Trade tolerance:         ±0.01 kW

Remaining shared constraint types:


,Rows
Trade reciprocity,96


## 16. Last: inspect GNE enumeration dimensions, optionally attempt small bounded exploration

This calls GNE solver with strict limits as well as runtime/space protections so program does not end up crashing; can be gradually increased if there has been success

In [20]:
import inspect
from p2p.setup import assemble_game, build_game, to_static_gnep
for fn in (assemble_game,build_game,to_static_gnep):
    print(f'{fn.__name__}: {inspect.getsourcefile(fn)}:{inspect.getsourcelines(fn)[1]}')

assemble_game: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/p2p/setup.py:123
build_game: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/p2p/setup.py:374
to_static_gnep: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/p2p/setup.py:495


In [21]:
import pandas as pd
T = gne_case.T
rows = []
for N in (4, 16, 32):
    rows.append({'N':N, 'T':T, 'approx_controls':11*N*T,
                 'shared_trade_rows':2*N*T, 'beta_dimension':(N-1)*2*N*T,
                 'approx_local_input_rows':32*N*T})
display(pd.DataFrame(rows).set_index('N'))
print('These are cyclic, ring-topology forecasts, not measured larger-population models.')
print('At N=16 or 32, feeder redundancy must be certified anew.')

,T,approx_controls,shared_trade_rows,beta_dimension,approx_local_input_rows
N,,,,,
4,12,528,96,288,1536
16,12,2112,384,5760,6144
32,12,4224,768,23808,12288


These are cyclic, ring-topology forecasts, not measured larger-population models.
At N=16 or 32, feeder redundancy must be certified anew.


In [ ]:
# Run AFTER building gne_case (24h, 2h, epsilon-relaxed, screened feeder)
import importlib, json, os, pickle, signal, subprocess, sys, textwrap, time
from datetime import datetime
from pathlib import Path
import psutil

REGION_LIMIT = 100
BETA = 2.0
TIME_LIMIT_MIN = 100
MAX_RAM_GB = min(12.0, round(psutil.virtual_memory().total / 1024**3 * 0.70, 1))
MIN_FREE_GB = 0

p2p_root = Path(importlib.import_module('p2p.setup').__file__).resolve().parents[1]
dynect_root = Path(importlib.import_module('dynect').__file__).resolve().parents[1]
run_dir = p2p_root / '.gne_experiments' / datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir.mkdir(parents=True, exist_ok=True)
input_path = run_dir / 'game.pkl'
output_path = run_dir / 'gne_set.pkl'
summary_path = run_dir / 'summary.json'
log_path = run_dir / 'solver.log'
worker_path = run_dir / 'worker.py'
with input_path.open('wb') as f:
    pickle.dump(gne_case, f)

worker_path.write_text(textwrap.dedent(r'''
    import importlib, json, pickle, sys, time
    from pathlib import Path
    import numpy as np
    from p2p.setup import to_static_gnep
    gm = importlib.import_module('dynect.solvers.gne')

    def log(*args):
        print(time.strftime('%H:%M:%S'), *args, flush=True)

    # Trace the two expensive phases without modifying the installed package.
    if hasattr(gm, '_julia') and hasattr(gm._julia, 'mpsolve'):
        real_mpsolve = gm._julia.mpsolve
        def traced_mpsolve(*args, **kwargs):
            log('ENTER Julia ParametricDAQP.mpsolve (region exploration)')
            t0 = time.monotonic()
            result = real_mpsolve(*args, **kwargs)
            log('EXIT Julia mpsolve after', round(time.monotonic()-t0,1), 's')
            return result
        gm._julia.mpsolve = traced_mpsolve
    if hasattr(gm, 'gne_regions'):
        real_filter = gm.gne_regions
        def traced_filter(*args, **kwargs):
            log('ENTER GNE-region filtering')
            t0 = time.monotonic()
            result = real_filter(*args, **kwargs)
            log('EXIT filtering after', round(time.monotonic()-t0,1), 's',
                'GNE regions:', len(result.regions))
            return result
        gm.gne_regions = traced_filter

    input_path, output_path, summary_path = map(Path, sys.argv[1:4])
    beta, limit = float(sys.argv[4]), int(sys.argv[5])
    log('Constructing StaticGNEP...')
    with input_path.open('rb') as f:
        case = pickle.load(f)
    t0 = time.monotonic()
    _, _, gnep = to_static_gnep(case)
    n_beta = (gnep.N-1)*gnep.m_sh
    log('StaticGNEP ready after', round(time.monotonic()-t0,1), 's:',
        'n=',sum(gnep.n), 'm_sh=',gnep.m_sh, 'beta=',n_beta)
    assert n_beta <= 300
    log('Starting GNESetSolver; verbose=2, region_limit=', limit)
    solver = gm.GNESetSolver(gnep,
        beta_lb=np.full(n_beta,-beta), beta_ub=np.full(n_beta,beta),
        region_limit=limit, verbose=2)
    summary = dict(status=str(solver.status), regions=len(solver.gne_set.regions),
                   beta_dimension=n_beta, limit=limit)
    summary_path.write_text(json.dumps(summary,indent=2))
    log('Solver returned:', summary)
    with output_path.open('wb') as f:
        pickle.dump(solver.gne_set,f)
    log('Saved region set:', output_path)
'''))

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([str(p2p_root),str(dynect_root),env.get('PYTHONPATH','')])
env['PYTHONUNBUFFERED'] = '1'
check = subprocess.run([sys.executable, '-c',
    'from p2p.setup import to_static_gnep; from dynect.solvers.gne import GNESetSolver'],
    cwd=p2p_root, env=env, capture_output=True, text=True)
if check.returncode:
    raise RuntimeError(check.stderr)

print(f'Start: region_limit={REGION_LIMIT}, beta=±{BETA}, '
      f'time={TIME_LIMIT_MIN} min, RAM watchdog={MAX_RAM_GB:.1f} GB')
print('Experiment folder:', run_dir, flush=True)
start = time.monotonic(); last_report = start; offset = 0
peak_gb = 0.0; stop_reason = None; cpu_seconds = 0.0
with log_path.open('w') as log:
    proc = subprocess.Popen(
        [sys.executable, '-u', str(worker_path), str(input_path),
         str(output_path), str(summary_path), str(BETA), str(REGION_LIMIT)],
        cwd=p2p_root, env=env, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True)
    while proc.poll() is None:
        time.sleep(2)
        try:
            p = psutil.Process(proc.pid)
            group = [p] + p.children(recursive=True)
            memory = 0; cpu_seconds = 0.0
            for child in group:
                try:
                    memory += child.memory_info().rss
                    c = child.cpu_times(); cpu_seconds += c.user + c.system
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    pass
            peak_gb = max(peak_gb, memory / 1024**3)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
        if peak_gb > MAX_RAM_GB:
            stop_reason = 'RAM watchdog'
        elif psutil.virtual_memory().available / 1024**3 < MIN_FREE_GB:
            stop_reason = 'Low system RAM'
        elif (time.monotonic()-start) > TIME_LIMIT_MIN*60:
            stop_reason = 'Time watchdog'
        if stop_reason:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            break
        if time.monotonic()-last_report >= 30:
            print(f'[{(time.monotonic()-start)/60:.1f} min] '
                  f'RAM peak {peak_gb:.2f} GB; CPU time {cpu_seconds:.0f} s', flush=True)
            with log_path.open(errors='replace') as f:
                f.seek(offset); chunk = f.read(); offset = f.tell()
            if chunk:
                print('New worker output:\n'+chunk[-2500:],flush=True)
            last_report = time.monotonic()
    proc.wait()
print(f'EXIT={proc.returncode}, elapsed={(time.monotonic()-start)/60:.1f} min, '
      f'peak RAM={peak_gb:.2f} GB, stop={stop_reason}', flush=True)
if summary_path.exists():
    print('RESULT:',summary_path.read_text())
else:
    with log_path.open(errors='replace') as f:
        f.seek(max(0,log_path.stat().st_size-4000)); print('LOG TAIL:\n',f.read())


Start: region_limit=100, beta=±2.0, time=100 min, RAM watchdog=5.6 GB
Experiment folder: /Users/catharinahente/Documents/Github/p2p-energy-trading-illustrative/p2p_dynect/.gne_experiments/20261122_100649
[0.5 min] RAM peak 0.74 GB; CPU time 28 s
New worker output:
10:06:52 Constructing StaticGNEP...
10:06:52 StaticGNEP ready after 0.1 s: n= 528 m_sh= 96 beta= 288
10:06:52 Starting GNESetSolver; verbose=2, region_limit= 100
10:06:52 ENTER Julia ParametricDAQP.mpsolve (region exploration)

[1.0 min] RAM peak 0.74 GB; CPU time 56 s
[1.5 min] RAM peak 0.74 GB; CPU time 81 s
[2.0 min] RAM peak 0.74 GB; CPU time 106 s
[2.5 min] RAM peak 0.74 GB; CPU time 132 s
[3.0 min] RAM peak 0.74 GB; CPU time 158 s
[3.5 min] RAM peak 0.74 GB; CPU time 187 s
[4.1 min] RAM peak 0.74 GB; CPU time 214 s
[4.6 min] RAM peak 0.74 GB; CPU time 243 s
[5.1 min] RAM peak 0.74 GB; CPU time 272 s
[5.6 min] RAM peak 0.74 GB; CPU time 301 s
[6.1 min] RAM peak 0.74 GB; CPU time 331 s
[6.6 min] RAM peak 0.74 GB; CPU time

# Analyse Resulting Regions 

In [ ]:
# Run only after the monitored worker has saved gne_set.pkl
import pickle
import numpy as np
import pandas as pd
from IPython.display import display
from p2p.setup import evaluated_costs, to_static_gnep

if not output_path.exists():
    print('No saved GNE set yet. The last timed-out run returned no regions.')
else:
    with output_path.open('rb') as f:
        gne_set = pickle.load(f)
    print('GNE regions returned:',len(gne_set.regions))
    print('Solver status:',summary_path.read_text() if summary_path.exists() else 'unknown')

    # Same epsilon-relaxed game as the enumerator, for a comparable vGNE.
    if 'v_epsilon' not in globals():
        v_epsilon = solve_compact(gne_case,verbose=False)
    J_v = evaluated_costs(gne_case,v_epsilon['u'],first_day=True)
    print('Epsilon-game vGNE costs [EUR]:',np.round(J_v,5))

    rows=[]; samples={}
    n_expected = gne_case.T*sum(gne_case.game.nu)
    for k,reg in enumerate(gne_set.regions,1):
        try:
            theta=np.asarray(reg.theta_feas,float).ravel()
            u=(np.asarray(reg.K,float) @ theta).ravel()+np.asarray(reg.h,float).ravel()
            if u.size!=n_expected:
                raise ValueError(f'Expected {n_expected} controls, got {u.size}')
            J=evaluated_costs(gne_case,u,first_day=True)
            violation=model_max_violation(gne_case,u)
        except (TypeError,ValueError,AttributeError) as exc:
            print(f'Region {k}: representative unavailable ({exc})')
            continue
        samples[k]=u
        row={'region':k, 'full_constraint_violation':violation,
             'total_cost_EUR':float(J.sum()), 'max_cost_EUR':float(J.max()),
             'cost_std_EUR':float(J.std()), 'max_vs_vGNE_EUR':float(np.max(J-J_v)),
             'RMS_dispatch_vs_vGNE_kW':float(np.sqrt(np.mean((u-v_epsilon['u'])**2)))}
        row.update({f'P{i+1}_EUR':float(ji) for i,ji in enumerate(J)})
        # Optional: provide SAME-RESOLUTION no-trade costs as J_nt_2h.
        if 'J_nt_2h' in globals():
            nt=np.asarray(J_nt_2h,float).ravel()
            assert nt.shape==J.shape
            gains=nt-J
            row['minimum_saving_EUR']=float(gains.min())
            row['worst_proportional_saving']=float(np.min(gains/np.maximum(abs(nt),1e-9)))
        rows.append(row)
    results=pd.DataFrame(rows)
    if results.empty:
        print('No region representatives could be evaluated.')
    else:
        print('First: feasible representatives with minimum cost dispersion')
        display(results.sort_values('cost_std_EUR').head(10).round(5))
        print('Best sampled total cost:')
        display(results.sort_values('total_cost_EUR').head(5).round(5))
        print('NOTE: each row is ONE feasible point per region, not a fairness optimum '
              'optimized over that entire region.')


unilateral best response gap computation (to check GNE feasibility)

In [ ]:
# Optional: confirm that representative schedules are GNEs of the epsilon game.
import gurobipy as gp
from gurobipy import GRB

if 'results' not in globals() or results.empty:
    print('No GNE representatives to verify yet.')
else:
    _,_,static_gne = to_static_gnep(gne_case)
    cuts=np.cumsum([0]+list(static_gne.n))
    def unilateral_regrets(u):
        xs=[np.asarray(u[cuts[i]:cuts[i+1]],float) for i in range(static_gne.N)]
        regrets=[]
        for i in range(static_gne.N):
            q=np.asarray(static_gne.q[i],float).ravel().copy()
            for j in range(static_gne.N):
                if j != i:
                    q+=np.asarray(static_gne.Q[i][j],float) @ xs[j]
            H=np.asarray(static_gne.Q[i][i],float)
            H=0.5*(H+H.T)
            rhs=np.asarray(static_gne.b_sh,float).ravel().copy()
            for j in range(static_gne.N):
                if j!=i:
                    rhs-=np.asarray(static_gne.A_sh[j],float) @ xs[j]
            with gp.Model() as m:
                m.Params.OutputFlag=0
                z=m.addMVar(static_gne.n[i],lb=-GRB.INFINITY)
                m.addConstr(np.asarray(static_gne.A_loc[i]) @ z <=
                            np.asarray(static_gne.b_loc[i]).ravel())
                m.addConstr(np.asarray(static_gne.A_sh[i]) @ z <= rhs)
                m.setObjective(0.5*z @ H @ z + q @ z, GRB.MINIMIZE)
                m.optimize()
                if m.Status!=GRB.OPTIMAL:
                    raise RuntimeError(f'Prosumer {i+1}: Gurobi status {m.Status}')
                current=0.5*xs[i]@H@xs[i]+q@xs[i]
                regrets.append(max(0.0,float(current-m.ObjVal)))
        return np.asarray(regrets)

    chosen=results.sort_values('cost_std_EUR').head(3)['region']
    for region_id in chosen:
        gaps=unilateral_regrets(samples[int(region_id)])
        print(f'Region {region_id}: unilateral cost improvement [EUR]: '
              f'{np.round(gaps,7)}; maximum={gaps.max():.3e}')
